# Speech Emotion Recognition - Colab training (free tier)

Dataset: **xbgoose/ravdess** - 1,440 clips, 24 actors, 8 emotions, ONE `train` split.

Run the cells **in order, top to bottom**. Each cell has a check that stops you
before wasting GPU time. Full instructions and error fixes: `process.txt`.

Expected time on a free T4: ~15-20 minutes for 6 epochs + augmentation.

## Step 1 - Check the runtime has a GPU
If this says CPU: `Runtime > Change runtime type > T4 GPU`. Training still works on
CPU but takes hours instead of minutes.

In [ ]:
import torch, sys, platform
print('python :', sys.version.split()[0])
print('torch  :', torch.__version__)
print('cuda   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu    :', torch.cuda.get_device_name(0))
    print('memory :', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
else:
    print('WARNING: no GPU. Runtime > Change runtime type > T4 GPU.')
print('machine:', platform.machine())

## Step 2 - Get the code
Replace the URL if you are working from a fork. If the repo is private you need a
token: `https://<TOKEN>@github.com/<user>/<repo>.git` (keep it out of shared notebooks).

In [ ]:
import os, pathlib
REPO = 'https://github.com/Apoorva-13S/Real_time_speech_emotion.git'
BRANCH = 'main'          # change if your work is on another branch
WORKDIR = '/content/repo'

if pathlib.Path(WORKDIR).is_dir():
    !cd {WORKDIR} && git fetch --all && git checkout {BRANCH} && git pull
else:
    !git clone --branch {BRANCH} --depth 1 {REPO} {WORKDIR}

os.chdir(WORKDIR)
%cd {WORKDIR}
print('working directory:', os.getcwd())
print('files:', sorted(p.name for p in pathlib.Path('.').iterdir())[:14])

## Step 3 - Install dependencies
Colab already has torch + transformers, so this mainly installs `datasets` and
`soundfile` and prints what versions you ended up with (write them down - they
are the first thing anyone will ask about if a result cannot be reproduced).

In [ ]:
!pip install -q -r requirements.txt
import transformers, datasets, librosa, soundfile, numpy
print('transformers', transformers.__version__)
print('datasets    ', datasets.__version__)
print('librosa     ', librosa.__version__)
print('soundfile   ', soundfile.__version__, '/ libsndfile', soundfile.__libsndfile_version__)
print('numpy       ', numpy.__version__)
assert int(numpy.__version__.split('.')[0]) < 2, 'numpy 2.x breaks older librosa; reinstall with numpy<2'
print('OK')

## Step 4 - Optional: mount Google Drive for checkpoint safety
Free Colab disconnects. If your checkpoints live on Drive, a disconnect costs you
minutes, not hours. Skip this cell if you do not want to sign in.

In [ ]:
USE_DRIVE = True
RUN_DIR = '/content/ser_runs/full'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_DIR = '/content/drive/MyDrive/ser_runs/full'
    os.makedirs('/content/drive/MyDrive/ser_runs', exist_ok=True)

os.makedirs(RUN_DIR, exist_ok=True)
print('checkpoints will be written to', RUN_DIR)

## Step 5 - Download and verify the dataset
This prints the row count, actors, per-emotion counts and durations. **Read the
output**: if any emotion shows 0, stop here rather than training a broken model.

Note: `xbgoose/ravdess` has ONE split named `train`. We split the 24 actors
ourselves in the next step. It does not come pre-split into 8 parts.

In [ ]:
import sys
sys.path.insert(0, 'src')

from ser.data import load_ravdess_hf
from ser.labels import CLASSES
import numpy as np

clips = load_ravdess_hf('xbgoose/ravdess')
actors = sorted({c.actor for c in clips})
counts = {c: sum(1 for k in clips if k.emotion == c) for c in CLASSES}
dur = [c.duration_s for c in clips]

print('clips   :', len(clips))
print('actors  :', len(actors))
print('duration: min %.2f  median %.2f  max %.2f s' % (min(dur), np.median(dur), max(dur)))
for c in CLASSES:
    print('  %-10s %4d' % (c, counts[c]))

assert len(clips) == 1440, f'expected 1440 clips, got {len(clips)}'
assert len(actors) == 24, f'expected 24 actors, got {len(actors)}'
assert all(v > 0 for v in counts.values()), f'missing classes: {[k for k,v in counts.items() if v==0]}'
print('\nDATASET OK')

## Step 6 - Train
Uses `configs/colab_train.yaml`: 6 epochs, batch 8, augmentation x4, warmup +
cosine schedule, GradScaler, class weights.

**Do not judge the model by the validation line alone** - the cell prints a test
report on actors the model never saw, plus macro F1 (which cannot hide a dead class).

In [ ]:
from ser import config
from ser.train import train

cfg = config.resolve('configs/colab_train.yaml')
cfg['run']['output_dir'] = RUN_DIR
cfg['data']['splits_file'] = RUN_DIR + '/data/splits.json'

result = train(cfg)
print('\nbest epoch:', result['best_epoch'])
print('test macro F1:', result['test']['macro_f1'])
print('test accuracy:', result['test']['accuracy'])

## Step 7 - Look at the results before trusting them
These three checks separate a real improvement from a re-labelled number:

1. **macro F1** - if this is near 0.3 while accuracy is 0.6, some classes are dead.
2. **per-class recall** - any class at 0.00 means the model cannot recognise it.
3. **beat the baseline** - the model must beat always predicting the most common class.

In [ ]:
from ser import metrics
import json, pathlib

print(metrics.format_report(result['test'], result['classes']))
print('\nmajority-class baseline:', result['majority_baseline'])

if result['test']['accuracy'] <= result['majority_baseline']:
    print('\n*** DO NOT SHIP: the model does not beat always guessing one class ***')
if result['test']['macro_f1'] < 0.35:
    print('*** WARNING: macro F1 below 0.35 - several classes are not learned ***')

figs = pathlib.Path(RUN_DIR) / 'reports' / 'figures'
print('\nfigures:', sorted(p.name for p in figs.glob('*.png')) if figs.is_dir() else 'none')

## Step 8 - Try the model on a file
Upload a WAV (2-6 s of speech) and see the prediction. This is the same code path
the web demo uses, so if it works here it will work there.

In [ ]:
from google.colab import files
from ser.predict import EmotionPredictor, format_prediction

pred = EmotionPredictor(RUN_DIR + '/best')
up = files.upload()
for name in up:
    preds = pred.predict_file(name)
    print(name, '->', format_prediction(preds, 3))
    for p in preds:
        print('   %-10s %5.1f%%  %s' % (p['label'], p['score']*100, '#'*int(p['score']*40)))

## Step 9 - Package the model for deployment
Creates `model.zip` containing the checkpoint plus `labels.json` and `metrics.json`.
Download it, then follow **process.txt > Publishing the model** to attach it to a
GitHub Release, which is what the live demo downloads.

In [ ]:
import shutil, pathlib, json

stage = pathlib.Path('/content/bundle')
if stage.exists():
    shutil.rmtree(stage)
stage.mkdir(parents=True)

shutil.copytree(pathlib.Path(RUN_DIR) / 'best', stage / 'model')
for extra in ('labels.json', 'metrics.json'):
    src = pathlib.Path(RUN_DIR) / extra
    if src.is_file():
        shutil.copy2(src, stage / extra)

shutil.make_archive('/content/model', 'zip', root_dir=stage)
size = pathlib.Path('/content/model.zip').stat().st_size / 1e6
print(f'/content/model.zip  ({size:.0f} MB)')
print('contents:', sorted(p.name for p in (stage / 'model').iterdir()))

from google.colab import files
files.download('/content/model.zip')

## Step 10 - Record the run (do not skip this)
Writes a small text file with everything needed to reproduce the result. Commit it
to the repo, or paste it into your report. Without it, the number in your README is
not reproducible.

In [ ]:
import json, platform, sys, pathlib, torch, transformers, datasets

record = {
    'dataset': 'xbgoose/ravdess',
    'n_clips': 1440, 'n_actors': 24, 'n_classes': len(result['classes']),
    'split': result['split'],
    'augmentation': result['augmentation'],
    'best_epoch': result['best_epoch'],
    'val': {k: result['val'][k] for k in ('accuracy', 'macro_f1', 'weighted_f1')},
    'test': {k: result['test'][k] for k in ('accuracy', 'macro_f1', 'weighted_f1')},
    'majority_baseline': result['majority_baseline'],
    'hardware': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    'env': {'python': sys.version.split()[0], 'torch': torch.__version__,
            'transformers': transformers.__version__, 'datasets': datasets.__version__},
}
out = pathlib.Path('/content/run_record.json')
out.write_text(json.dumps(record, indent=2))
print(out.read_text())
files.download(str(out))